In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI
import json

: 

In [ ]:
meta = pd.read_csv("../data/Project_JOB_meta_data.csv")
schedule = pd.read_csv("../data/Project_Schedule_Data.csv")

meta.head()

In [ ]:
meta.columns = meta.columns.str.lower()
schedule.columns = schedule.columns.str.lower()

meta.fillna("", inplace=True)
schedule.fillna("", inplace=True)

In [ ]:
numeric_cols = ["total_control_estimate","building_size"]

categorical_cols = [
    "region_name",
    "business_unit",
    "office_name",
    "city",
    "state",
    "core_market",
    "market_sector",
    "project_type"
]

In [ ]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
])

pipeline = Pipeline([
    ("preprocess", preprocessor)
])

matrix = pipeline.fit_transform(meta)

In [ ]:
new_project = {
    "region_name":"Southwest",
    "business_unit":"Construction",
    "office_name":"Phoenix",
    "city":"Phoenix",
    "state":"AZ",
    "core_market":"Commercial",
    "market_sector":"Office",
    "project_type":"New Build",
    "total_control_estimate":20000000,
    "building_size":80000
}

new_df = pd.DataFrame([new_project])

In [ ]:
new_vec = pipeline.transform(new_df)

similarity = cosine_similarity(new_vec, matrix)[0]

meta["similarity_score"] = similarity

top_projects = meta.sort_values("similarity_score",ascending=False).head(5)

top_projects

In [ ]:
project_ids = top_projects["control_job_no"].tolist()

similar_schedules = schedule[
    schedule["main_projectid"].isin(project_ids)
]

similar_schedules.head()

In [ ]:
similar_schedules["plannedstartdate"] = pd.to_datetime(similar_schedules["plannedstartdate"])
similar_schedules["plannedfinishdate"] = pd.to_datetime(similar_schedules["plannedfinishdate"])

similar_schedules["duration_days"] = (
    similar_schedules["plannedfinishdate"]
    - similar_schedules["plannedstartdate"]
).dt.days

In [ ]:
schedule_summary = similar_schedules.groupby("wbsname").agg(
    avg_duration=("duration_days","mean"),
    task_frequency=("wbsname","count")
).reset_index()

schedule_summary = schedule_summary.sort_values("task_frequency",ascending=False)

schedule_summary.head(20)

In [ ]:
prompt = f"""
You are a construction scheduling expert.

New Project:
{json.dumps(new_project,indent=2)}

Historical schedule patterns:
{schedule_summary.head(20).to_json(orient="records")}

Generate a recommended project schedule.

Return JSON format:

[
{{"phase":"","task":"","duration_days":"","predecessor":""}}
]
"""

In [ ]:
client = OpenAI(api_key="YOUR_API_KEY")

response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {"role":"system","content":"You are a construction planner."},
        {"role":"user","content":prompt}
    ],
    temperature=0.2
)

schedule = response.choices[0].message.content

print(schedule)